In [ ]:
Inputs. Hidden. Output.

In [426]:
import torch
import torch.nn.functional as F
import random
torch.set_printoptions(sci_mode=False)

words = open("names.txt", "r").read().splitlines()
random.shuffle(words)

d_h = 10
d_emb = 10
d_V = 27


alphabet = list("abcdefghijklmnopqrstuvwxyz")
stoi = {alphabet[i] : i + 1 for i in range(26)}
stoi["."] = 0
itos = {i : letter for letter, i in stoi.items()}
vocab_size = len(alphabet) + 1

In [448]:
initial_hidden = torch.randn(d_h, 1)*0.01
U = torch.randn(d_h, d_h)
W = torch.randn(d_h, d_emb)
V = torch.randn(d_V, d_h)*0.01

b_h = torch.randn(d_h, 1)*0.0001
b_y = torch.randn(d_V, 1)*0.0001

emb = torch.randn(d_emb, d_V)*0.01

params = [U, W, V, emb, initial_hidden, b_h, b_y]

for p in params:
    p.requires_grad = True

In [449]:
def one_hot(X, num_classes=None):
    length = X+1 if num_classes is None else num_classes
    out = torch.zeros(length, 1)
    out[X] = 1
    return out

def cross_entropy(output, target):
    output_max = torch.max(output)
    output = output - output_max
    output_exp = torch.exp(output)
    output_exp_sum = output_exp.sum()
    output = output_exp/output_exp_sum
    out = (output*target).sum()
    out = -torch.log(out)
    return out

In [450]:
kwargs = {
    "lr_arch" : "momentum based",
    "beta1": 0.9,
    "beta2": 0.9,
    "eps" : 1e-6,
    "gamma" : 0.9,
    "momentum" : 0.1,
    "learn" : 0.01,
    "batch_size" : 1,
    "iterations" : len(words)
}

In [451]:
def train_model(lr_arch = None, momentum = 0.1, beta1 = 0.9, beta2 = 0.9 ,gamma = 0.9, eps = 1e-5, learn = 0.1, batch_size = 32, iterations = 1000):
    for j in range(iterations):
        # ix = torch.randint(0, len(words), (batch_size,))
        # ix = [j]
        word = words[j]
        hidden_layer = initial_hidden
        losses = []
        word = "." + word + "."
        X = word[:-1]
        Y = word[1:]

        if lr_arch == "momentum based":
            velocities = [torch.zeros_like(p) for p in params]
        if lr_arch == "adagrad":
            accum_grad = [torch.zeros_like(p) for p in params]
        if lr_arch == "rmsprop":
            accum_grad_exp = [torch.zeros_like(p) for p in params]
        if lr_arch ==  "adam":
            accum_mt = [torch.zeros_like(p) for p in params]
            accum_vt = [torch.zeros_like(p) for p in params]

        for i in range(len(X)):
            letter = X[i]
            idx = stoi[letter]
            e = emb @ one_hot(idx, num_classes=d_V)
            hidden_layer = torch.tanh(W @ e + U @ hidden_layer + b_h)
            y = V @ hidden_layer + b_y
            losses.append(cross_entropy(y, one_hot(stoi[Y[i]], num_classes=27)))

        for p in params:
            p.grad = None

        loss = torch.stack(losses).mean()
        loss.backward()
        if j % 1000 == 0:
            print(f"{j}/{iterations} | {loss}")

        lr = learn
        for i in range(len(params)):
            if lr_arch == "adam":
                accum_mt[i] = beta1*accum_mt[i] + (1-beta1)*params[i].grad
                accum_vt[i] = beta2*accum_vt[i] + (1-beta2)*params[i].grad**2
                t = j + 1
                mt_hat = accum_mt[i]/(1-beta1**t)
                vt_hat = accum_vt[i]/(1-beta2**t)
                update = (mt_hat/(vt_hat**0.5 + eps)) * lr

            elif lr_arch == "rmsprop":
                accum_grad_exp[i] = gamma*accum_grad_exp[i] + (1-gamma)*params[i].grad**2
                update = (params[i].grad/(accum_grad_exp[i] + eps)**0.5) * lr

            elif lr_arch == "momentum based":
                velocities[i] = lr*params[i].grad + momentum*velocities[i]
                update = velocities[i]

            elif lr_arch == "adagrad":
                accum_grad[i] = accum_grad[i] + params[i].grad**2
                update = (params[i].grad/(accum_grad[i] + eps)**0.5) * lr
            else:
                update = lr*params[i].grad

            params[i].data += -update

In [452]:
train_model(**kwargs)

0/32033 | 3.3023314476013184
1000/32033 | 2.7133190631866455
2000/32033 | 2.619744062423706
3000/32033 | 2.5926620960235596
4000/32033 | 2.300999402999878
5000/32033 | 2.552218437194824
6000/32033 | 2.613161563873291
7000/32033 | 2.1101229190826416
8000/32033 | 2.2203545570373535
9000/32033 | 1.8379439115524292
10000/32033 | 2.504020929336548
11000/32033 | 2.7395052909851074
12000/32033 | 1.6575497388839722
13000/32033 | 3.1382079124450684
14000/32033 | 2.233931541442871
15000/32033 | 2.455958604812622
16000/32033 | 2.3280389308929443
17000/32033 | 2.409898042678833
18000/32033 | 2.5863826274871826
19000/32033 | 3.2543060779571533
20000/32033 | 2.19101619720459
21000/32033 | 2.4546709060668945
22000/32033 | 2.244245767593384
23000/32033 | 2.4206812381744385
24000/32033 | 2.3893444538116455
25000/32033 | 2.75483775138855
26000/32033 | 3.5353283882141113
27000/32033 | 1.7750471830368042
28000/32033 | 3.3488974571228027
29000/32033 | 2.8732452392578125
30000/32033 | 2.820145845413208
3100

In [458]:
y = 0
set_words = set(words)
i = 0
num = 10
for _ in range(num):
    hidden_layer = initial_hidden
    word = ""
    while True:
        e = emb @ one_hot(y, num_classes=d_V)
        hidden_layer = torch.tanh(W @ e + U @ hidden_layer)
        y = V @ hidden_layer
        y = torch.multinomial(torch.softmax(y, dim = 0).view(-1), num_samples=1).item()
        word += itos[y]
        if itos[y] == ".":
            if word not in set_words:
                print(word)
            if word in set_words:
                i += 1

            break
print(i/num*100)

niured.
elzo.
fokata.
dasnlesl.
zuvye.
jasgy.
morse.
mawdena.
mubecy.
osya.
0.0


In [460]:
hidden_layer = initial_hidden
e = emb @ one_hot(0, num_classes=d_V)
hidden_layer = torch.tanh(W @ e + U @ hidden_layer)
y = V @ hidden_layer
probs = torch.softmax(y, dim=0).view(-1)
print(probs)
print(itos[probs.argmax().item()], probs.max().item())

tensor([0.0010, 0.0975, 0.0478, 0.0604, 0.0537, 0.0337, 0.0224, 0.0306, 0.0178,
        0.0110, 0.0849, 0.0872, 0.0346, 0.0792, 0.0502, 0.0161, 0.0245, 0.0080,
        0.0374, 0.0626, 0.0375, 0.0070, 0.0195, 0.0170, 0.0108, 0.0099, 0.0375],
       grad_fn=<ViewBackward0>)
a 0.09749193489551544
